In [16]:
import pandas as pd

df = pd.read_csv('Project2_dataset.csv')

df.info()

<class 'pandas.DataFrame'>
RangeIndex: 57301 entries, 0 to 57300
Data columns (total 9 columns):
 #   Column                            Non-Null Count  Dtype
---  ------                            --------------  -----
 0   Airline Name                      57301 non-null  str  
 1   Airline Country                   57301 non-null  str  
 2   Departure Airport Name            57301 non-null  str  
 3   Departure Airport City            57301 non-null  str  
 4   Departure Airport Country/Region  57301 non-null  str  
 5   Arrival Airport Name              57301 non-null  str  
 6   Arrival Airport City              57301 non-null  str  
 7   Arrival Airport Country/Region    57301 non-null  str  
 8   Plane Name                        57301 non-null  str  
dtypes: str(9)
memory usage: 3.9 MB


In [17]:
corrections = {
    'Alberto Carnevalli Airport': 'Venezuela',
    'Charles M. Schulz Sonoma County Airport': 'United States',
    'Cheddi Jagan International Airport': 'Guyana',
    'Cibao International Airport': 'Dominican Republic',
    'Comodoro Arturo Merino Benitez International Airport': 'Chile', 
    'Eugene F. Correira International Airport': 'Guyana',
    'El Alto International Airport': 'Bolivia',
    'F. D. Roosevelt Airport': 'Netherlands Antilles',
    'Futuna Airport': 'Vanuatu',
    'General Jose Antonio Anzoategui International Airport': 'Venezuela',
    'JAGS McCartney International Airport': 'Turks and Caicos Islands',
    'London City Airport': 'United Kingdom',
    'London Gatwick Airport': 'United Kingdom',
    'London Heathrow Airport': 'United Kingdom',
    'London Luton Airport': 'United Kingdom',
    'London Stansted Airport': 'United Kingdom',
    'Luis Munoz Marin International Airport': 'Puerto Rico',
    'Mayor Buenaventura Vivas International Airport': 'Venezuela',
    'Norman Manley International Airport': 'Jamaica',
    'Norman Y. Mineta San Jose International Airport': 'United States',
    'Northwest Florida Beaches International Airport': 'United States',
    'Presidente Joao Batista Figueiredo Airport': 'Brazil',
    'St Petersburg Clearwater International Airport': 'United States',
    'St Pierre Airport': 'Saint Pierre and Miquelon',    
    'Sydney Kingsford Smith International Airport': 'Australia',
    'Albany Airport': 'Australia', 
    'Alexandria International Airport': 'United States',
    'Atlas Brasil Cantanhede Airport': 'Brazil',
    'Arturo Michelena International Airport': 'Venezuela',
    'Birmingham-Shuttlesworth International Airport': 'United States',
    'Cochin International Airport': 'India',
    'Florence Regional Airport': 'United States',
    'Fort Smith Regional Airport': 'United States',
    'Richmond Airport': 'Australia',
    'San Jose Airport': 'Philippines',
    'Santa Ana Airport': 'Solomon Islands',
    'Santa Fe Municipal Airport': 'United States',
    'Santa Rosa International Airport': 'Ecuador',
    'Tri-Cities Regional TN/VA Airport': 'United States',
    'Victoria Regional Airport': 'United States',
    'Waterloo Regional Airport': 'United States'
}

for airport, correct_country in corrections.items():
    
    df.loc[
        df['Departure Airport Name'].str.startswith(airport, na=False), 
        'Departure Airport Country/Region'
    ] = correct_country
    
    df.loc[
        df['Arrival Airport Name'].str.startswith(airport, na=False), 
        'Arrival Airport Country/Region'
    ] = correct_country

In [18]:

dep_airports = df[['Departure Airport Name', 'Departure Airport City', 'Departure Airport Country/Region']].rename(columns={
    'Departure Airport Name': 'name',
    'Departure Airport City': 'city',
    'Departure Airport Country/Region': 'country'
})

arr_airports = df[['Arrival Airport Name', 'Arrival Airport City', 'Arrival Airport Country/Region']].rename(columns={
    'Arrival Airport Name': 'name',
    'Arrival Airport City': 'city',
    'Arrival Airport Country/Region': 'country'
})

airports_df = pd.concat([dep_airports, arr_airports]).drop_duplicates(subset=['name', 'city', 'country']).dropna(subset=['name'])

airports_df.to_csv('airports_nodes.csv', index=False)

In [19]:
airlines_df = df[['Airline Name', 'Airline Country']].rename(columns={
    'Airline Name': 'name',
    'Airline Country': 'country'
}).drop_duplicates(subset=['name']).dropna(subset=['name'])

airlines_df.to_csv('airlines_nodes.csv', index=False)

In [20]:
# Custom function to handle the semicolon-separated Plane Name column
def get_unique_planes(plane_series):
    all_planes_str = ';'.join(plane_series.dropna())
    unique_planes = list(set(all_planes_str.split(';')))
    unique_planes = [p.strip() for p in unique_planes if p.strip()]
    return ';'.join(unique_planes)

# Custom function to get unique airlines
def get_unique_airlines(airline_series):
    unique_airlines = list(set(airline_series.dropna()))
    return ';'.join(unique_airlines)

# Group and aggregate
routes_df = df.groupby(['Departure Airport Name', 'Arrival Airport Name']).agg(
    airline_names=('Airline Name', get_unique_airlines),
    aircraft_types=('Plane Name', get_unique_planes)
).reset_index()

# Rename to clean keys
routes_df.rename(columns={
    'Departure Airport Name': 'departure_airport',
    'Arrival Airport Name': 'arrival_airport'
}, inplace=True)

routes_df = routes_df[routes_df['departure_airport'] != routes_df['arrival_airport']].reset_index(drop=True)

routes_df.to_csv('routes_edges.csv', index=False)